# Markdown passages, chunks, and citations

Converts the inventory-selected Markdown into exact source passages and retrieval-ready chunks. This notebook does not build BM25, embeddings, or model inference.

In [1]:
from pathlib import Path
import json, sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').exists()), None)
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root')
sys.path.insert(0, str(ROOT / 'src'))
from mobile_rag.corpus import build_chunks, chunk_statistics, export_chunks, latest_inventory_manifest, validate_chunks
inventory_path = latest_inventory_manifest(ROOT)
print({'project_root': str(ROOT), 'inventory': inventory_path.relative_to(ROOT).as_posix()})

{'project_root': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag', 'inventory': 'artifacts/step-02/2026-09-13T07-57-37+00-00_d6d0852754/corpus_manifest.json'}


In [2]:
bundle = build_chunks(ROOT, inventory_path)
checks = validate_chunks(bundle, ROOT)
stats = chunk_statistics(bundle)
print({'documents': len(bundle['documents']), 'passages': len(bundle['passages']), 'chunks': len(bundle['chunks']), 'exceptions': len(bundle['exceptions'])})
print(stats)
print(checks)

{'documents': 14, 'passages': 22586, 'chunks': 4355, 'exceptions': 1035}
{'count': 4355, 'minimum_chars': 44, 'median_chars': 929, 'p95_chars': 2913, 'maximum_chars': 8311, 'oversized_count': 980, 'pdf_mapping_unavailable_count': 0, 'context_characters': 256903}
{'passed': True, 'checks': {'unique_passage_ids': True, 'unique_chunk_ids': True, 'source_slices_exact': True, 'nonwhitespace_source_coverage': True, 'segment_links_valid': True, 'retrieval_segments_exact': True, 'neighbor_links_valid': True, 'no_cross_document_chunks': True, 'source_hashes_unchanged': True, 'all_documents_have_chunks': True}, 'scope': 'structural preservation only; not clinical/OCR validation'}


In [3]:
kind_counts = {}
for passage in bundle['passages']:
    kind_counts[passage['kind']] = kind_counts.get(passage['kind'], 0) + 1
kind_counts

{'heading': 38,
 'table': 30,
 'page_marker': 2336,
 'paragraph': 13132,
 'list': 6990,
 'html_or_opaque': 60}

In [4]:
examples = []
for chunk in bundle['chunks'][:5]:
    examples.append({
        'chunk_id': chunk['chunk_id'],
        'document_id': chunk['document_id'],
        'characters': chunk['character_count'],
        'pages': chunk['declared_pages'],
        'flags': chunk['flags'],
        'preview': chunk['retrieval_text'][:300],
    })
examples

[{'chunk_id': 'chunk_3dc64ac39513658e687c',
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'characters': 152,
  'pages': [],
  'flags': [],
  'preview': '# Bubble-CPAP Guidelines (2017)\r\n\n\n| Field | Value |\r\n| --- | --- |\r\n| Source | `Bubble-CPAP-guidelines-2017.pdf` |\r\n| Pages | 10 |\r\n| OCR pages | 0 |\r\n'},
 {'chunk_id': 'chunk_d4830842cfb25755a732',
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'characters': 212,
  'pages': [2],
  'flags': [],
  'preview': '# Guidelines for Use of Bubble-CPAP Concentrators\r\n\n\nStandard oxygen therapy, either using concentrators or cylinders, saves the lives of many children. Use CPAP if the child is failing standard oxygen therapy.\r\n'},
 {'chunk_id': 'chunk_fdf02387d4d3e92528b8',
  'document_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1',
  'characters': 416,
  'pages': [2],
  'flags': [],
  'preview': '# Guidelines

In [5]:
oversized = sorted((c for c in bundle['chunks'] if 'oversized' in c['flags']), key=lambda c: c['character_count'], reverse=True)
[{'chunk_id': c['chunk_id'], 'characters': c['character_count'], 'pages': c['declared_pages']} for c in oversized[:10]]

[{'chunk_id': 'chunk_a527905c675e1c874ee6', 'characters': 8311, 'pages': [7]},
 {'chunk_id': 'chunk_46f2d16b34e5e8710bfe', 'characters': 8175, 'pages': [8]},
 {'chunk_id': 'chunk_b709f3924b9d1cb43a66', 'characters': 7668, 'pages': [6]},
 {'chunk_id': 'chunk_7e88a22ea66601edac76', 'characters': 6533, 'pages': [20]},
 {'chunk_id': 'chunk_a8bcaa77f4171d67fba0', 'characters': 6483, 'pages': [37]},
 {'chunk_id': 'chunk_a2bc7f4f4912bc91463d', 'characters': 6358, 'pages': [51]},
 {'chunk_id': 'chunk_b700809ecadea31528f2', 'characters': 6261, 'pages': [34]},
 {'chunk_id': 'chunk_62e121732546e948b008', 'characters': 6179, 'pages': [52]},
 {'chunk_id': 'chunk_5dd148ad3cb0f0915c37', 'characters': 6121, 'pages': [15]},
 {'chunk_id': 'chunk_5e3c36f01eb8f1b8e824', 'characters': 6065, 'pages': [40]}]

In [6]:
second = build_chunks(ROOT, inventory_path)
assert bundle['run']['bundle_fingerprint'] == second['run']['bundle_fingerprint']
assert [c['chunk_id'] for c in bundle['chunks']] == [c['chunk_id'] for c in second['chunks']]
output_dir = export_chunks(bundle, ROOT)
saved_checks = json.loads((output_dir / 'check_results.json').read_text(encoding='utf-8'))
assert saved_checks['passed']
print({'output_dir': output_dir.relative_to(ROOT).as_posix(), 'bundle_fingerprint': bundle['run']['bundle_fingerprint'], 'deterministic': True, 'checks_passed': True})

{'output_dir': 'artifacts/step-05/2026-09-13T08-01-20+00-00_9f352c688e', 'bundle_fingerprint': '9f352c688e71637f0478088b17cea2ac52cbf46f90abde4120f63a6198cfcf68', 'deterministic': True, 'checks_passed': True}


## Checkpoint

The outputs establish structural preservation and citation traceability against existing Markdown. They do not establish OCR accuracy, clinical correctness, retrieval quality, or model performance. Continue with the retrieval notebook after reviewing these outputs.